# Encapsulation and `property` in Python

**Encapsulation is a key concept in object-oriented programming**.

The idea is to distinguish between an object's **public interface** —the parts other objects and functions are expected to use— and its **internal implementation details**.

Some languages provide strict modifiers such as `public`, `protected`, and `private`. Python takes a different approach: it relies much more on **programming conventions** than on absolute access restrictions.

## 1. Public attributes

A public attribute can be accessed freely from outside the object.

In Python, any attribute whose name does not begin with an underscore is considered public by convention.

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

p = Person("Ana")
print(p.name)
p.name = "Marta"
print(p.name)

This is completely normal in Python.

There is no need to automatically create `get_name()` and `set_name()` methods for every attribute, as is common in some other languages.

## 2. So-called protected attributes: `_attribute`

When a name begins with a single underscore:

```python
_attribute
```

Python communicates an intention:

> This attribute is intended for internal use by the class or its subclasses.

However, **Python does not prevent access from outside**.

In [ ]:
class Person:
    def __init__(self, name):
        self._name = name

p = Person("Ana")
print(p._name)   # This is possible
p._name = "Luis"
print(p._name)

Therefore, `_name` is not truly protected.

The underscore is a **convention**: it tells other programmers that they should not depend directly on this attribute unless there is a good reason.

It is sometimes called *protected* by analogy with other languages, but Python does not enforce strict protection.

## 3. Double-underscore attributes: `__attribute`

If an attribute begins with two underscores:

```python
__attribute
```

Python applies a mechanism called **name mangling**.

These attributes are sometimes informally described as "private", but they are **not private in a strict sense**.

In [ ]:
class Person:
    def __init__(self, name):
        self.__name = name

p = Person("Ana")
# This raises AttributeError:
# print(p.__name)

It may look as if `__name` is truly private, but Python does not completely hide it.

Internally, Python transforms the name to reduce accidental collisions, especially in inheritance scenarios.

In [ ]:
class Person:
    def __init__(self, name):
        self.__name = name

p = Person("Ana")
print(p._Person__name)

So an attribute beginning with `__` **is still accessible** if we know the mangled name.

The main purpose of `__attribute` is not security or absolute access prevention, but to **reduce accidental access and avoid name collisions**.

## 4. Do `public`, `protected`, and `private` really exist?

In Python, the usual conventions can be summarized as follows:

| Convention | Example | Usual meaning |
|---|---|---|
| Public | `name` | Part of the normal public interface |
| Internal use | `_name` | Should not normally be used directly from outside |
| Name mangling | `__name` | Reduces accidental access and name collisions |

Therefore:

- `_attribute` is **not protected** by the language;
- `__attribute` is **not strictly private**;
- Python relies heavily on programmers respecting the interface designed by a class.

## 5. Why use encapsulation then?

Even though Python allows access to almost everything, encapsulation is still useful.

It helps us:

- separate the public interface from implementation details;
- validate values before modifying object state;
- preserve object invariants;
- change the internal implementation without forcing client code to change;
- make code easier to understand and maintain.

## 6. How do we control access or modification then?

If Python does not strictly prevent access to attributes, a natural question is:

> How can we control which values are assigned or what happens when an attribute is accessed?

This is where `property` becomes especially useful.

A `property` **does not make an attribute private**. Its main purpose is to preserve natural attribute syntax while adding behavior, for example:

- validating values;
- computing a value dynamically;
- preventing certain assignments;
- changing the internal implementation without changing client code.

## 7. Traditional getters and setters

We could write a class like this:

In [ ]:
class Person:
    def __init__(self, age):
        self._age = age

    def get_age(self):
        return self._age

    def set_age(self, age):
        if age < 0:
            raise ValueError("Age cannot be negative")
        self._age = age

This style works, but in Python it is often unnecessarily verbose.

Python provides `property`, which lets us control access while keeping natural attribute syntax.

## 8. `@property`

`property` lets us define a method that is used from outside the class with **attribute syntax**.

This is useful when we want to keep a simple interface such as:

```python
p.age
```

while still executing code whenever that value is accessed.

In [ ]:
class Person:
    def __init__(self, age):
        self._age = age

    @property
    def age(self):
        return self._age

p = Person(20)
print(p.age)

Although `age` is implemented using a method, from outside it is used like an attribute:

```python
p.age
```

and not like:

```python
p.age()
```

## 9. Property setter

We can also control assignments using `@name.setter`.

In [ ]:
class Person:
    def __init__(self, age):
        self._age = 0
        self.age = age

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("Age cannot be negative")
        self._age = value

p = Person(20)
print(p.age)
p.age = 25
print(p.age)
# p.age = -5  # would raise ValueError

Here, `age` is part of the **public interface**, while `_age` is an **internal implementation detail**.

The property does not make `_age` inaccessible. The important point is that the class defines `age` as the normal way to interact with that data.

The setter validates values without requiring users of the class to call an explicit method such as `set_age()`.

## 10. Read-only properties

If we define `@property` but do not define a setter, the property cannot be assigned directly.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14159 * self.radius ** 2

c = Circle(2)
print(c.area)
# c.area = 10  # would raise AttributeError

In this case, `area` looks like an attribute, but it is actually computed every time it is accessed.

## 11. An important advantage of `property`

We can start with a simple public attribute:

In [ ]:
class Person:
    def __init__(self, age):
        self.age = age

Later, if validation or additional behavior becomes necessary, we can convert it into a `property` **without changing how the rest of the program uses it**:

```python
p.age
p.age = 30
```

This is an important reason why Python programmers generally do not create trivial getters and setters from the beginning "just in case".

We can start with a public attribute and introduce a `property` only when it becomes useful.

## 12. Encapsulation in Python: a practical view

In Python, encapsulation does not necessarily mean **physically preventing access** to data.

It mainly means:

- defining a clear public interface;
- distinguishing that interface from internal implementation details;
- controlling modifications when necessary;
- preserving object invariants;
- allowing the internal implementation to change without affecting client code.

Encapsulation in Python depends both on language features and on programmers respecting conventions and the interface designed by the class.

## 13. Summary

- `attribute` → public by convention.
- `_attribute` → indicates internal use, but remains accessible.
- `__attribute` → uses *name mangling*; it does not provide strict privacy.
- Python relies more on conventions and interface design than on rigid access restrictions.
- `@property` **does not make an attribute private**: it adds behavior while preserving attribute syntax.
- `@property_name.setter` lets us validate or control assignments.
- A property without a setter can act as a read-only attribute.
- Trivial getters and setters are usually unnecessary from the beginning.
- Encapsulation is mainly about separating a stable public interface from internal implementation details.

# Exercises

## Exercise 1

Implement a `BankAccount` class with an internal `_balance` attribute and a read-only `balance` property.

In [ ]:
class BankAccount:
    # YOUR CODE HERE
    pass

## Exercise 2

Implement a `Product` class with a `price` property.

The price must always be greater than or equal to `0`. If a negative value is assigned, raise `ValueError`.

In [ ]:
class Product:
    # YOUR CODE HERE
    pass

## Exercise 3

Implement a `Temperature` class with a `celsius` property and a read-only `fahrenheit` property.

Use:

```python
fahrenheit = celsius * 9 / 5 + 32
```

In [ ]:
class Temperature:
    # YOUR CODE HERE
    pass